# Bank Loan Total Return Swaps in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Portfolio and transactions |
| 4 | Valuation and accrued interest |
| 5 | Instrument events |

## The instrument

A total return swap has two legs pulling in opposite directions: one side carries the return on a
**leveraged loan** (the asset leg), and the other pays the cost of financing that exposure (the
funding leg):

    asset leg     the referenced loan, via ReferenceInstrument
    funding leg   a FloatingLeg paying an index plus a spread

The loan itself is a `FlexibleLoan`. Its schedule can be fixed or floating -- here it's fixed.

## A constraint on this setup

**The portfolio's holding recipe has to be set at creation -- you can't add it later.** Any
portfolio that holds, or even just references, a `FlexibleLoan` needs a recipe attached when it's
created, so LUSID's valuation engine has something to resolve against. There's no way to attach a
recipe to an existing portfolio afterwards, so a portfolio created without one has to be deleted
and rebuilt.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

The loan's rate history runs through four contiguous periods, each with its own coupon. The
valuation date lands inside the second period, after one step-up has already kicked in. The swap
itself only resets once, at maturity -- the loan's own periods already carry all the rate detail
it needs.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "BankLoanTrsDemo"
RECIPE    = "bank-loan-trs-demo-recipe"
PORTFOLIO = "bank-loan-trs-demo-book"

LOAN_ID   = "DEMO-NWB-LOAN-01"
LOAN_DESC = "Demo Northwind Broadcasting Term Loan B"
SWAP_ID   = "DEMO-NWB-TRS-01"
SWAP_DESC = "Demo Northwind Broadcasting Term Loan B TRS"
CURRENCY  = "USD"

START     = d(2025, 3, 1)
MATURITY  = d(2030, 3, 1)
ASOF      = d(2026, 6, 15)

# (period start, annual coupon) -- contiguous periods; the last runs to MATURITY.
RATE_STEPS = [
    (d(2025, 3, 1), 0.0800),
    (d(2026, 3, 1), 0.0825),
    (d(2027, 3, 1), 0.0850),
    (d(2028, 3, 1), 0.0875)]

TENOR      = "3M"
DAY_COUNT  = "Actual360"

FIN_INDEX  = "SOFRRATE"
FIN_SPREAD = 0.0275          # 275bp
FIN_RATE   = 0.0530          # the fixing this notebook supplies for the current period

# The funding leg resets every TENOR (3M) from START, in arrears -- LUSID needs a fixing quote
# dated at the reset date that opens the period containing ASOF, not at START or at ASOF itself.
# START -> 2025-03-01, 2025-06-01, ... -> 2026-06-01 is the reset that opens ASOF's period.
FUNDING_RESET = d(2026, 6, 1)

STRIKE      = 97.00          # points -- the swap's initial mark
LOAN_MARK   = 98.50          # points -- the loan's current mark
DENOM       = 100

INITIAL_PRICE = STRIKE / DENOM
NOTIONAL      = 1.0          # every FixedSchedule row on the loan shares this notional

QUANTITY = 40_000.00

CURRENT_RATE = next(rate for reset, rate in reversed(RATE_STEPS) if reset <= ASOF)

print(f"{SWAP_DESC}")
print(f"  asset   Receive  {LOAN_ID}")
for reset, rate in RATE_STEPS:
    flag = "  <- in force" if rate == CURRENT_RATE else ""
    print(f"      {reset:%Y-%m-%d}  {rate:.3%}{flag}")
print(f"  funding Pay      {FIN_INDEX} + {FIN_SPREAD:.2%}")
print(f"  {QUANTITY:,.0f} units, strike {STRIKE} -> mark {LOAN_MARK} on {ASOF:%Y-%m-%d}")
print(f"  asset leg return = {QUANTITY:,.0f} x ({LOAN_MARK} - {STRIKE}) / {DENOM} = "
      f"{QUANTITY * (LOAN_MARK - STRIKE) / DENOM:,.2f} {CURRENCY}")

Demo Northwind Broadcasting Term Loan B TRS
  asset   Receive  DEMO-NWB-LOAN-01
      2025-03-01  8.000%
      2026-03-01  8.250%  <- in force
      2027-03-01  8.500%
      2028-03-01  8.750%
  funding Pay      SOFRRATE + 2.75%
  40,000 units, strike 97.0 -> mark 98.5 on 2026-06-15
  asset leg return = 40,000 x (98.5 - 97.0) / 100 = 600.00 USD


---
# 1. Instrument creation

## 1a. The loan

The loan gets mastered as its own instrument rather than embedded inside the swap, because the
asset leg points at it by identifier, not by value.

In [3]:
bounds = []
for i in range(len(RATE_STEPS) - 1):
    bounds.append((RATE_STEPS[i][0], RATE_STEPS[i + 1][0], RATE_STEPS[i][1]))
bounds.append((RATE_STEPS[-1][0], MATURITY, RATE_STEPS[-1][1]))

schedules = [
    m.FixedSchedule(
        schedule_type="FixedSchedule",
        start_date=start,
        maturity_date=end,
        flow_conventions=m.FlowConventions(
            currency=CURRENCY,
            payment_frequency=TENOR,
            day_count_convention=DAY_COUNT,
            roll_convention=str(start.day),
            payment_calendars=[], reset_calendars=[]),
        coupon_rate=rate,
        notional=NOTIONAL,
        payment_currency=CURRENCY,
        stub_type="ShortBack")
    for start, end, rate in bounds]

loan = m.FlexibleLoan(
    instrument_type="FlexibleLoan",
    start_date=START,
    maturity_date=MATURITY,
    dom_ccy=CURRENCY,
    schedules=schedules)

LOAN_LUID = upsert("loan", LOAN_DESC, LOAN_ID, loan)
print(f"Loan : {LOAN_LUID}")

Loan : LUID_00003DFV


## 1b. The swap

The asset leg holds a `ReferenceInstrument` that points at the loan's `ClientInternal` identifier
. Its `reset_schedule` fires just once, at maturity.

In [4]:
trs = m.TotalReturnSwap(
    instrument_type="TotalReturnSwap",
    start_date=START,
    maturity_date=MATURITY,
    asset_leg=m.AssetLeg(
        asset=m.ReferenceInstrument(
            instrument_type="ReferenceInstrument",
            instrument_id=LOAN_ID,
            instrument_id_type="ClientInternal",
            scope=SCOPE),
        pay_receive="Receive",
        initial_price=INITIAL_PRICE,
        reset_schedule=m.ResetSchedule(frequency=TENOR, first_reset_date=MATURITY),
        income_policy="Reinvest"),
    funding_leg=m.FloatingLeg(
        instrument_type="FloatingLeg",
        start_date=START,
        maturity_date=MATURITY,
        notional=INITIAL_PRICE,
        leg_definition=m.LegDefinition(
            rate_or_spread=FIN_SPREAD,
            pay_receive="Pay",
            conventions=m.FlowConventions(
                currency=CURRENCY,
                payment_frequency=TENOR,
                day_count_convention=DAY_COUNT,
                roll_convention=str(START.day),
                payment_calendars=[], reset_calendars=[]),
            index_convention=m.IndexConvention(
                currency=CURRENCY,
                payment_tenor=TENOR,
                fixing_reference=FIN_INDEX,
                index_name=FIN_INDEX,
                publication_day_lag=0,
                day_count_convention=DAY_COUNT),
            reset_convention="InArrears",
            stub_type="ShortBack",
            notional_exchange_type="None")))

TRS_LUID = upsert("trs", SWAP_DESC, SWAP_ID, trs)
print(f"Bank loan TRS : {TRS_LUID}")

Bank loan TRS : LUID_00003DFW


---
# 2. Recipe

This recipe uses `FlexibleLoanPricer`, registered against `TotalReturnSwap` since that's what's
actually held here in the portfolio. The `ReferenceInstrument` on the asset leg resolves to the loan's
`LusidInstrumentId` before pricing looks up a quote, so the loan's quote needs to be keyed by
that LUID, not by its `ClientInternal` identifier. The funding leg's index, meanwhile, is
resolved as a `Rate` quote against a `RIC` identifier.

In [5]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Bank loan TRS, FlexibleLoanPricer",
            market=m.MarketContext(
                market_rules=[
                    m.MarketDataKeyRule(
                        key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                        quote_type="Price", field="mid", quote_interval="5D"),
                    m.MarketDataKeyRule(
                        key="Quote.ClientInternal.*", supplier="Lusid", data_scope=SCOPE,
                        quote_type="Price", field="mid", quote_interval="5D"),
                    m.MarketDataKeyRule(
                        key="Quote.RIC.*", supplier="Lusid", data_scope=SCOPE,
                        quote_type="Rate", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_scope=SCOPE,
                    default_instrument_code_type="ClientInternal")),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="FlexibleLoanPricer",
                    instrument_type="TotalReturnSwap")],
                options=m.PricingOptions(
                    model_selection=m.ModelSelection(
                        library="Lusid", model="FlexibleLoanPricer"),
                    use_instrument_type_to_determine_pricer=True,
                    allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: BankLoanTrsDemo/bank-loan-trs-demo-recipe


---
# 3. Portfolio and transactions

`instrument_event_configuration.recipe_id` has to be set at creation, as covered above --
`recreate_portfolio()`'s `recipe=` argument is what sets it.

Since the swap is entered into rather than bought, `totalConsideration` comes in at zero.

In [6]:
recreate_portfolio(PORTFOLIO, "Bank Loan TRS Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-TRS",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": TRS_LUID},
        transaction_date=START.isoformat(),
        settlement_date=START.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, START, START))

Recreated BankLoanTrsDemo/bank-loan-trs-demo-book


,date,type,luid,units,consideration
0,2025-03-01,Buy,LUID_00003DFW,"40,000.00",0.00


---
# 4. Valuation and accrued interest

Valuation here needs two quotes: the loan's own mark (a `Price` quote against its LUID), and the
funding index's current fixing (a `Rate` quote against its `RIC` identifier), dated at the reset
that opened the period `ASOF` falls in, not at `ASOF` itself.

`Valuation/PV` is the swap's full economics, both legs together. `Valuation/Leg1/PV` isolates the
asset leg's own price return on its own, handy for checking it against the loan's own mark move.
`Valuation/Accrued` is scoped more narrowly than `PV`: it's the loan's own coupon accrued since its last coupon date.

In [7]:
def upsert_loan_price(price, effective):
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{LOAN_LUID}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=LOAN_LUID,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=CURRENCY))})


def upsert_fixing(rate, effective):
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{FIN_INDEX}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=FIN_INDEX,
                    instrument_id_type="RIC",
                    quote_type="Rate", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=rate, unit=CURRENCY))})


upsert_loan_price(LOAN_MARK / DENOM, ASOF)
upsert_fixing(FIN_RATE, FUNDING_RESET)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/Leg1/PV",       "Sum"),
           ("Valuation/PV",            "Sum"),
           ("Valuation/Accrued",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

leg1_pv = result.loc[result["Instrument/default/Name"] == SWAP_DESC,
                     "Sum(Valuation/Leg1/PV)"].iloc[0]
print(f"LUSID Leg1/PV {leg1_pv:,.2f}  vs  asset leg return "
      f"{QUANTITY * (LOAN_MARK - STRIKE) / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/Leg1/PV),Sum(Valuation/PV),Sum(Valuation/Accrued)
0,Demo Northwind Broadcasting Term Loan B TRS,"40,000.00",600.00,"-11,149.26",128.33


LUSID Leg1/PV 600.00  vs  asset leg return 600.00


---
# 5. Instrument events

The `instrument_event_configuration.recipe_id` we set back in section 3 is needed wherever a
`FlexibleLoan` shows up in the dependency graph, before `FlexibleLoanPricer` can value anything
against the portfolio. That same setting also lets you forecast the swap's own events -- skip it,
and `query_applicable_instrument_events` just comes back with zero events instead of raising an
error.

The `TotalReturnSwap` itself carries a `MaturityEvent` at `MATURITY`, and that needs no separate
configuration. Query a window that spans past that date and you get the event back with a
transaction already populated -- there's no transaction type to register beforehand.

In [8]:
events_api = api(lusid.InstrumentEventsApi)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=ASOF.isoformat(),
        window_end=(MATURITY + timedelta(days=5)).isoformat(),
        effective_at=(MATURITY + timedelta(days=5)).isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

,event type,eligible balance,status
0,MaturityEvent,"40,000.00",Active


---
# Summary

1. A bank loan TRS points at its loan through a `ReferenceInstrument`, referencing a separately
   mastered `FlexibleLoan` rather than holding it inline.
2. A `FlexibleLoan`'s schedule can be fixed or floating -- this one's fixed.
3. The swap prices under `FlexibleLoanPricer`, registered here against `TotalReturnSwap` since
   that's what's actually held -- the same model would register against `FlexibleLoan` for a loan
   held directly.
4. The portfolio's holding recipe can only be set at creation time. Any `FlexibleLoan` in the
   dependency graph needs one attached, or there's nothing for valuation to resolve against.
5. That same create-time recipe is also what makes the swap's own `MaturityEvent` forecastable --
   `query_applicable_instrument_events` needs it, just like `FlexibleLoanPricer` does.

In [9]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Loan       : {LOAN_LUID}")
print(f"Swap       : {TRS_LUID}")

Scope      : BankLoanTrsDemo
Portfolio  : BankLoanTrsDemo/bank-loan-trs-demo-book
Recipe     : BankLoanTrsDemo/bank-loan-trs-demo-recipe
Loan       : LUID_00003DFV
Swap       : LUID_00003DFW
